# Section 7 · End-to-End Agent Evaluation (Groq edition)

> **Groq edition.** The Bedrock notebook next to this one scores the agent with AgentCore Evaluations, whose judges are Bedrock models. Here the agent still runs on **AgentCore Runtime**, but its model and the judge both run on **Groq**, and the evaluators come from the open-source [Strands Evals SDK](https://github.com/strands-agents/evals) — the same metrics (Goal Success Rate, Correctness, Helpfulness) plus the same custom travel-quality judge. Strands Evals reads the agent's traces from CloudWatch, just as AgentCore Evaluations does.

The complete evaluation lifecycle in one notebook:

1. **Deploy** a travel assistant agent to AgentCore Runtime
2. **Create** the evaluators, including a custom LLM-as-judge
3. **Invoke** the agent with in-scope and out-of-scope prompts
4. **Evaluate** the session with built-in and custom evaluators
5. **Summarise** and save the results

### The agent
A travel assistant (`agent_app_groq.py`) that answers questions about ✈️ flights, 🏨 hotels and 🌤️ weather — and should **refuse** anything else. The custom evaluator checks exactly that.

| | |
|:--|:--|
| Agent framework | Strands Agents |
| Model | `openai/gpt-oss-120b` on Groq |
| Deployment | AgentCore Runtime (CodeBuild — no local Docker) |
| Evaluation | Strands Evals, judged by a Groq model |

### Prerequisites
- A `GROQ_API_KEY` in your environment or a `.env` file
- AWS credentials; IAM permissions for `bedrock-agentcore:*`, `ecr:*`, `iam:CreateRole`/`PassRole`, `codebuild:*`, `s3:*`, `logs:*`, `xray:*` — no Bedrock model access needed
- **CloudWatch Transaction Search** turned on (once per account and Region). The agent's traces are read from there. The launch step below tries to enable it automatically; if it can't, enable it in the CloudWatch console (*Application Signals → Transaction Search*).

---
## Step 0 — Install dependencies

In [ ]:
%pip install -q -r requirements-groq.txt

---
## Step 1 — Imports and AWS session

In [ ]:
import json
import os
import time
import uuid

import boto3
from boto3.session import Session
from dotenv import load_dotenv
from IPython.display import Markdown, display

from bedrock_agentcore_starter_toolkit import Runtime

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

boto_session = Session()
region = boto_session.region_name or "us-east-1"
account_id = boto3.client("sts", region_name=region).get_caller_identity()["Account"]

print(f"Region  : {region}")
print(f"Account : {account_id}")

---
## Step 2 — Deploy the travel agent to AgentCore Runtime

The starter toolkit's `Runtime` class configures the agent, builds the container with CodeBuild, and deploys it.

> ⏱️ First deployment takes ~5–10 minutes. On a new account, add up to ~10 minutes while Transaction Search activates.

In [ ]:
runtime = Runtime()

runtime.configure(
    entrypoint="agent_app_groq.py",
    agent_name="s07_travel_agent_groq",
    requirements_file="agent_requirements-groq.txt",
    region=region,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    idle_timeout=120,
)

# The deployed agent reads GROQ_API_KEY and GROQ_MODEL_ID (see agent_app_groq.py).
# Passing the key as an environment variable is fine for a lab; in production keep it
# in Secrets Manager (or an AgentCore Identity API-key provider) and read it at startup.
AGENT_MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

launch_result = runtime.launch(
    auto_update_on_conflict=True,
    env_vars={"GROQ_API_KEY": os.environ["GROQ_API_KEY"], "GROQ_MODEL_ID": AGENT_MODEL_ID},
)
print(f"\nDeployment started: {launch_result.agent_arn}")

In [ ]:
def wait_for_ready(runtime, name="Agent", poll_seconds=15):
    """Poll until the agent runtime reaches a terminal status."""
    terminal = {"READY", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"}
    while True:
        status = runtime.status().endpoint["status"]
        print(f"  {name} status: {status}")
        if status in terminal:
            return status
        time.sleep(poll_seconds)


status = wait_for_ready(runtime, name="Travel Agent")
assert status == "READY", f"Deployment failed with status: {status}"
print("\n✅ Agent is READY")
print(f"   Agent ID  : {launch_result.agent_id}")
print(f"   Agent ARN : {launch_result.agent_arn}")

# Traces are stored only once CloudWatch Transaction Search is ACTIVE. On a new account the launch above has
# just enabled it, and activation can take ~10 minutes; traces sent before then are lost, so wait here.
xray = boto3.client("xray", region_name=region)
while (destination := xray.get_trace_segment_destination())["Status"] != "ACTIVE":
    print(f"  Transaction Search: {destination['Destination']} {destination['Status']}, waiting...")
    time.sleep(30)
assert destination["Destination"] == "CloudWatchLogs", "Turn on Transaction Search in the CloudWatch console"
print("✅ Transaction Search is ACTIVE")

---
## Step 3 — Create the evaluators

All evaluators are LLM-as-judge, and the judge is a Groq model:

- **Built-in** — `GoalSuccessRateEvaluator` (session level: did the user achieve their goals?), `CorrectnessEvaluator` and `HelpfulnessEvaluator` (trace level: is the answer accurate and useful?).
- **Custom travel quality** — the same judging prompt and 5-point scale as the AgentCore version, read from `travel_quality_metric.json` and given to an `OutputEvaluator` as its rubric. Any answer outside the travel domain gets **Very Poor (0.0)**.

In [ ]:
from strands.models.openai import OpenAIModel
from strands_evals.evaluators import CorrectnessEvaluator, GoalSuccessRateEvaluator, HelpfulnessEvaluator, OutputEvaluator

JUDGE_MODEL_ID = os.getenv("GROQ_JUDGE_MODEL_ID", "openai/gpt-oss-120b")
judge = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=JUDGE_MODEL_ID,
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Judge model: {JUDGE_MODEL_ID}")

In [ ]:
with open("travel_quality_metric.json") as f:
    judge_config = json.load(f)["llmAsAJudge"]

# The same instructions and rating scale as the AgentCore evaluator. Strands Evals passes the
# prompt and the answer to the judge itself, so the {context}/{assistant_turn} placeholders go.
instructions = judge_config["instructions"].split("\n\nContext:")[0]
scale = "\n".join(f"- {r['value']:.2f} ({r['label']}): {r['definition']}" for r in judge_config["ratingScale"]["numerical"])
rubric = f"{instructions}\n\nScore with exactly one of these values:\n{scale}"
# The original rubric only covers an assistant that ANSWERS an out-of-scope question. gpt-oss usually
# declines instead, so say explicitly that a polite refusal is the right behaviour.
rubric += (
    "\n\nIf the user asks about a non-travel topic and the assistant politely declines and steers back to travel, "
    "that is the correct behaviour: score it Very Good (1.00)."
)

travel_quality = OutputEvaluator(rubric=rubric, model=judge, name="travel_response_quality")
print(rubric)

---
## Step 4 — Invoke the agent with test prompts

All prompts share one `session_id`, so they form one conversation that we can evaluate as a whole.

In [ ]:
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

test_prompts = [
    "What flights are available from New York to London?",   # in scope
    "Can you recommend a mid-range hotel in Paris?",         # in scope
    "What is the weather like in London right now?",         # in scope
    "Can you help me write a Python script to sort a list?", # out of scope
]

responses = []
for prompt in test_prompts:
    print(f"📤 Prompt: {prompt}")
    response = runtime.invoke(payload={"prompt": prompt}, session_id=session_id)
    print(f"🤖 Response: {response.get('response', response)}\n")
    responses.append({"prompt": prompt, "response": response})

print(f"✅ Invoked {len(test_prompts)} prompts in session {session_id}")

---
## Step 5 — Wait for traces to be ingested

AgentCore Runtime sends OpenTelemetry traces to CloudWatch. Give them ~3 minutes to arrive before evaluating.

In [ ]:
TRACE_INGESTION_WAIT = 180  # seconds

print(f"Waiting {TRACE_INGESTION_WAIT}s for traces to be ingested...")
for remaining in range(TRACE_INGESTION_WAIT, 0, -30):
    print(f"  {remaining}s remaining...")
    time.sleep(30)
print("✅ Ready to evaluate")

---
## Step 6 — Run evaluations

Strands Evals fetches the session's traces from the agent's CloudWatch log group (`CloudWatchProvider`) and hands them to the evaluators. AgentCore Runtime keeps each turn's messages as events inside its spans, so a small subclass copies them out where `CloudWatchProvider` looks, and `GenericGenAISessionMapper` turns the spans into a session.

| Pass | Evaluator | Level | Measures |
|------|-----------|-------|----------|
| A | `GoalSuccessRateEvaluator` | Session | Did the user achieve their goals? |
| B | `CorrectnessEvaluator` + `HelpfulnessEvaluator` | Trace (the latest turn) | Accuracy and usefulness |
| C | custom travel quality | Each prompt | Scope adherence + quality |

In [ ]:
from strands_evals import Case, Experiment
from strands_evals.mappers import GenericGenAISessionMapper
from strands_evals.providers import CloudWatchProvider


class AgentCoreRuntimeProvider(CloudWatchProvider):
    """CloudWatchProvider for AgentCore Runtime spans, which carry the conversation as span events.

    CloudWatchProvider reads the messages only from separate event records, so copy each span's events out.
    """

    @staticmethod
    def _parse_query_results(results):
        records = CloudWatchProvider._parse_query_results(results)
        return records + [
            {"eventName": event["name"], "spanId": record["spanId"], "attributes": event.get("attributes", {})}
            for record in records
            for event in record.get("events", [])
        ]


provider = AgentCoreRuntimeProvider(
    log_group=f"/aws/bedrock-agentcore/runtimes/{launch_result.agent_id}-DEFAULT",
    region=region,
    mapper=GenericGenAISessionMapper(),  # turns spans with gen_ai.* message events into a Session
)
all_reports = {}  # pass title -> EvaluationReport (one row per case and evaluator)


async def evaluate(title, evaluators, cases, task):
    """Run the evaluators on the cases, show each result, and keep the report for the summary."""
    # Jupyter already runs an event loop, so await the async runner (run_evaluations() calls asyncio.run).
    # max_workers=1 judges one case at a time, which keeps the judge inside Groq's free-tier rate limit.
    report = await Experiment(cases=cases, evaluators=evaluators).run_evaluations_async(task, max_workers=1)
    all_reports[title] = report
    print(f"=== {title} ===")
    for case, score, reason in zip(report.cases, report.scores, report.reasons):
        emoji = "✅" if score >= 0.75 else ("⚠️" if score >= 0.5 else "❌")
        display(Markdown(f"{emoji} **{case['evaluator']}** · {case.get('name')} — {score:.2f}\n\n{reason}"))
    return report

### Pass A — Session level: Goal Success Rate

In [ ]:
session_case = [Case(name="travel_session", input=session_id)]
goal_results = await evaluate(
    "Goal Success Rate (session)",
    [GoalSuccessRateEvaluator(model=judge)],
    session_case,
    lambda case: provider.get_evaluation_data(case.input),
)

### Pass B — Trace level: Correctness and Helpfulness

In [ ]:
quality_results = await evaluate(
    "Correctness & Helpfulness (latest turn)",
    [CorrectnessEvaluator(model=judge), HelpfulnessEvaluator(model=judge)],
    session_case,
    lambda case: provider.get_evaluation_data(case.input),
)

### Pass C — Custom evaluator: travel quality and scope

One case per prompt, scored on the answer the agent gave in Step 4. Watch the out-of-scope Python question:

- If the agent **answered** it, the custom judge scores it **Very Poor (0.0)** — even if the built-in evaluators rated the answer as fine.
- If the agent **declined** (gpt-oss usually does), the custom judge scores the refusal **Very Good**, while the built-in evaluators in Pass B, which don't know the agent's scope, mark the same refusal as unhelpful.

Either way, generic metrics can't tell a scope violation from a correct refusal; an evaluator that encodes your agent's scope can.

In [ ]:
answers = {r["prompt"]: "".join(r["response"]["response"]) for r in responses}  # the answer arrives as text chunks
prompt_cases = [Case(name=f"prompt_{i + 1}", input=prompt) for i, prompt in enumerate(answers)]
custom_results = await evaluate(
    "Travel quality — custom (per prompt)",
    [travel_quality],
    prompt_cases,
    lambda case: answers[case.input],
)

---
## Step 7 — Summary table

In [ ]:
print(f"{'Evaluator':<36} {'Case':<16} {'Score':>6}")
print("-" * 70)
scores = []
for report in all_reports.values():
    for case, score in zip(report.cases, report.scores):
        scores.append(score)
        print(f"{case['evaluator']:<36} {str(case.get('name')):<16} {score:>6.2f}  {'█' * int(score * 10)}")

if scores:
    print("-" * 70)
    print(f"{'Average score':<36} {'':<16} {sum(scores) / len(scores):>6.2f}")

---
## Step 8 — Save results to a file

In [ ]:
os.makedirs("eval_output", exist_ok=True)

with open("eval_output/travel_agent_eval_results.json", "w") as f:
    json.dump({title: report.to_dict() for title, report in all_reports.items()}, f, indent=2, default=str)
print("✅ Results saved to eval_output/travel_agent_eval_results.json")

---
## Step 9 — Clean up

In [ ]:
result = runtime.destroy(delete_ecr_repo=True)
print("Removed:", *result.resources_removed, sep="\n  - ")

---
## What you just did

1. **Deployed** a Strands agent to AgentCore Runtime (no Docker on your machine), with its model on Groq
2. **Created** a custom LLM-as-judge evaluator, judged by a Groq model
3. **Invoked** the agent with in-scope and out-of-scope prompts
4. **Evaluated** the session with built-in and custom evaluators
5. **Verified** the custom evaluator judges scope correctly — failing an out-of-scope answer, rewarding a refusal — where generic metrics can't